# LUTM-1: self-extending Python evaluator

This notebook executes every physical LUTM transition with the fixed transition table. Its tape is a self-extending Python dictionary: there is no space budget or blank padding. A run starts from `p#x`, and after physical halting only the binary prefix immediately to the right of `#` is decoded.

Run this notebook with **`LUTM-1` as the working directory**.

In [ ]:
from itertools import product
from pathlib import Path
from time import perf_counter

required_files = ("lutm.py", "programs.py", "data/transition_table.csv")
missing = [name for name in required_files if not (Path.cwd() / name).is_file()]
if missing:
    raise RuntimeError(
        "Run this notebook from the LUTM-1 repository root; missing: "
        + ", ".join(missing)
    )

from lutm import ScalarUTMSimulator
from programs import PROGRAMS, get_program

simulator = ScalarUTMSimulator()
print(f"Loaded {len(simulator.table.states)} states and "
      f"{len(simulator.table.transitions):,} transitions.")

## Run one registered program

Choose a name from `PROGRAMS`, an input, and a transition budget. The program is placed directly on the tape without padding.

In [ ]:
program_name = "plus_one"
input_bits = "1011"
t_max = 10_000_000

registered = get_program(program_name)
if not registered.accepts(input_bits):
    raise ValueError(
        f"Input {input_bits!r} is outside the contract for {program_name!r}."
    )

started = perf_counter()
result = simulator.run(registered.program, input_bits, t_max=t_max)
elapsed = perf_counter() - started
expected = registered.target(input_bits) if registered.oracle is not None else None

print(f"task:        {program_name}")
print(f"program bits:{len(registered.program):>8,}")
print(f"input:       {input_bits!r}")
print(f"output:      {result.output!r}")
print(f"expected:    {expected!r}")
print(f"halted:      {result.halted}")
print(f"invalid:     {result.invalid} ({result.invalid_reason.name})")
print(f"T:           {result.T:,}")
print(f"space L/R:   {result.left_space_used:,} / {result.right_space_used:,}")
print(f"final head:  {result.final_head:,}")
print(f"elapsed:     {elapsed:.4f} s")

if result.invalid:
    raise RuntimeError(f"LUTM run failed: {result.invalid_reason.name}")
if expected is not None and result.output != expected:
    raise AssertionError(
        f"Wrong output for {program_name}({input_bits}): "
        f"got {result.output!r}, expected {expected!r}"
    )

## Exhaustively verify registered tasks

Set `selected_tasks` to a list of registered names, or to the string `"all"` for every registered program that has an oracle. Inputs are enumerated by bit length, then filtered through each task's declared input contract.

The square witness is intentionally very expensive in physical transitions. Selecting `"all"`, or adding `"square"`, can therefore require a much larger `task_t_max` and a long run.

In [ ]:
selected_tasks = ["identity_short", "bit_not", "plus_one", "times_two"]
# selected_tasks = "all"
min_input_length = 1
max_input_length = 4
task_t_max = 10_000_000
progress_every = 25

oracle_names = [name for name, item in PROGRAMS.items() if item.oracle is not None]
if selected_tasks == "all":
    task_names = oracle_names
else:
    if isinstance(selected_tasks, str):
        raise TypeError('selected_tasks must be a list of names or the string "all"')
    task_names = list(selected_tasks)
    unknown = [name for name in task_names if name not in PROGRAMS]
    if unknown:
        raise KeyError(f"Unknown registered tasks: {unknown}")
    without_oracle = [name for name in task_names if PROGRAMS[name].oracle is None]
    if without_oracle:
        raise ValueError(f"Tasks have no comparison oracle: {without_oracle}")
if min_input_length < 0 or max_input_length < min_input_length:
    raise ValueError("Require 0 <= min_input_length <= max_input_length.")
if progress_every < 1:
    raise ValueError("progress_every must be positive.")

print("Selected:", ", ".join(task_names))
print(f"Input lengths: {min_input_length}..{max_input_length}")

In [ ]:
def binary_inputs(min_length, max_length):
    for length in range(min_length, max_length + 1):
        for bits in product("01", repeat=length):
            yield "".join(bits)

verification_rows = []
suite_started = perf_counter()

for task_index, name in enumerate(task_names, start=1):
    item = get_program(name)
    accepted_inputs = [
        bits
        for bits in binary_inputs(min_input_length, max_input_length)
        if item.accepts(bits)
    ]
    print(
        f"[{task_index}/{len(task_names)}] {name}: "
        f"{len(accepted_inputs)} accepted inputs",
        flush=True,
    )
    task_started = perf_counter()
    max_T = 0
    for case_index, bits in enumerate(accepted_inputs, start=1):
        run = simulator.run(item.program, bits, t_max=task_t_max)
        target = item.target(bits)
        if run.invalid:
            raise RuntimeError(
                f"{name}({bits!r}) failed after T={run.T:,}: "
                f"{run.invalid_reason.name}"
            )
        if run.output != target:
            raise AssertionError(
                f"{name}({bits!r}) returned {run.output!r}; "
                f"expected {target!r}"
            )
        max_T = max(max_T, run.T)
        if case_index % progress_every == 0:
            print(f"  verified {case_index}/{len(accepted_inputs)}", flush=True)
    task_elapsed = perf_counter() - task_started
    verification_rows.append((name, len(accepted_inputs), max_T, task_elapsed))
    print(f"  passed in {task_elapsed:.3f} s; max T={max_T:,}", flush=True)

print(f"\nAll selected tasks passed in {perf_counter() - suite_started:.3f} s.")
print(f"{'task':<22} {'cases':>8} {'max T':>14} {'seconds':>10}")
for name, cases, max_T, seconds in verification_rows:
    print(f"{name:<22} {cases:>8,} {max_T:>14,} {seconds:>10.3f}")